# New Sythnetic Diarization

In [2]:
from glob import glob
from nanodrz.constants import CACHE_DIR
from dataclasses import dataclass, field
import torchaudio
import torch
import random
from os import path

paths = glob(path.join(CACHE_DIR, "utts", "*"))


@dataclass
class Utterance:
    file: str
    speaker: str
    segements: list[tuple[int, int]] = field(default_factory=lambda: [])

    @property
    def length(self):
        return self.segments[-1][-1]

In [3]:
utts = [torch.load(p) for p in paths]
snames = set([u.speaker for u in utts])
speakers = {s: [] for s in snames}

for u in utts:
    if u.speaker not in speakers:
        speakers[u.speaker] = []
    speakers[u.speaker].append(u)


for s in speakers.keys():
    utts: list[Utterance] = speakers[s]
    utts.sort(key=lambda x: x.segements[-1][-1])
    speakers[s] = utts

speakers.keys()

dict_keys(['2285', '3483', '424', '1839', '5538', '4674', '3088', '3925', '9342', '153', '3906', '4234', '4145', '1184', '1286', '439', '3148', '1795', '3728', '3437', '371', '3375', '3557', '3252', '288', '4263', '3712', '3141', '1392', '1365', '1944', '89', '52', '1649', '979', '246', '613', '538', '1973', '4486', '658', '198', '1433', '400', '261', '3347', '2925', '4552', '5007', '3979', '2591', '205', '1905', '826', '3135', '874', '4948', '4211', '263', '3858', '4993', '5405', '3243', '451', '1175', '5259', '60', '1963', '4780', '2674', '4310', '293', '399', '2425', '1553', '511', '3835', '324', '3266', '3736', '1685', '2349', '3330', '4009', '4532', '543', '3083', '1251', '1060', '1323', '3775', '472', '2574', '3668', '2280', '1827', '166', '5258', '2104', '1870', '3301', '3567', '4295', '7171', '150', '2270', '177', '271', '4025', '2388', '5129', '3296', '573', '3119', '2224', '612', '2446', '3723', '1413', '2262', '3744', '2724', '2923', '3072', '11528', '922', '2501', '362', '6

In [4]:
lens = [
    sum([u.segements[-1][-1] / 16000 for u in speakers[s]]) / len(speakers[s])
    for s in speakers.keys()
]

In [1]:
def artificial_diarisation_sample(
    speakers: dict[str, Utterance] = None,
    max_secs=30,
    min_secs=7.5,
    interrupt_max=0.2,
    silence_max=0.2,
    num_speakers=4,
    sr=16000,
    **kwargs,
):
    audio = torch.zeros(1, 0)
    names, labels = [], []

    cur_speakers = random.sample(speakers.keys(), k=random.randint(2, num_speakers))
    seconds = random.uniform(min_secs, max_secs - 1)

    last_speaker = None

    for i in range(20):
        # Pick a random speaker
        speaker: str = random.choice(cur_speakers)

        cur_len = audio.shape[-1] / sr

        if seconds - cur_len < 1:
            break

        if speaker == last_speaker:
            continue

        # Determine the pause / interrupt length
        intpad = random.uniform(interrupt_max, silence_max)

        # We might not have a sample of this length or
        max_sample_len = seconds - cur_len + intpad

        max_sample_len = max(max_sample_len, 1)

        utts = speakers[speaker]
        max_sample_len = min(max([x.length for x in utts]) / sr, max_sample_len)

        # What is our smallest segment potential
        sample_len = random.uniform(0.5, max_sample_len)

        # Pick a utterance that is as long as this or longer
        utts = list(filter(lambda x: x.length > max_sample_len, speakers[speaker]))
        utt: Utterance = random.choice(utts)

        # Choose a random start point that gives space for the length
        # pick a segment starting with speech
        starts, ends = list(zip(*utt.segements))
        starts = [s for s in starts if s < utt.length - max_sample_len]
        start = random.choice(starts)

        ends

        # Find an end point that is less than or equal to target seconds
        # Load the audio with this offset
        # Derive the labels from the segment labels

        random_sample = random_sample.sum(dim=0)[None]

        int_range = min(
            interrupt_max, audio.shape[-1] / sr, random_sample.shape[-1] / sr
        )
        sil_max = silence_max

        if last_speaker is not None and last_speaker.name == speaker.name:
            int_range = 0

        cut_point = int(random.uniform(-int_range, sil_max) * sr)
        start_label = audio.shape[-1] / sr + cut_point / sr

        padding = torch.zeros(1, random_sample.shape[-1] + cut_point)
        audio = torch.cat((audio, padding), dim=-1)
        audio[:, -random_sample.shape[-1] :] += random_sample

        if speaker.name not in names:
            i = len(names)
            names.append(speaker.name)
        else:
            i = names.index(speaker.name)

        name_label = chr(ord("A") + i)

        labels.append(
            [start_label, start_label + random_sample.shape[-1] / sr, name_label]
        )

        last_speaker = speaker

    return audio, labels

NameError: name 'Utterance' is not defined